# Filter Midnight Events

Remove synthetic or placeholder midnight punches while preserving an audit CSV of removed rows.

**Requires:** `events/events.csv`.  
**Produces:** `events.cleaned.csv`, removed rows, and a filter report.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("filter_midnight")
display(nb.pipeline_overview(ctx, "filter_midnight"))


## Controls


In [ ]:
VERBOSE = True
MAX_REMOVED_EXAMPLES = int(step_cfg.get("max_removed_examples_per_file", 10))
MAX_REMOVED_EXAMPLES


## Input Preview


In [ ]:
display(nb.artifact_table({"raw events": paths.events_csv}))
display(nb.preview_csv(paths.events_csv))


## Build Options


In [ ]:
from core.events.filtering.options import FilterMidnightEventsOptions

options = FilterMidnightEventsOptions(
    input_dir=str(paths.events_dir),
    report_json=str(paths.filter_report),
    removed_csv=str(paths.removed_midnight_csv),
    max_removed_examples_per_file=MAX_REMOVED_EXAMPLES,
    verbose=VERBOSE,
)
options


## Run Filter


In [ ]:
from core.drive.logging_utils import setup_logging
from core.events.filtering.service import run_from_options

setup_logging(VERBOSE)
filter_report = run_from_options(options)
display(nb.report_summary(filter_report))


## Compare Outputs


In [ ]:
display(nb.artifact_table({
    "cleaned events": paths.cleaned_events_csv,
    "removed midnight events": paths.removed_midnight_csv,
    "filter report": paths.filter_report,
}))
display(nb.preview_csv(paths.cleaned_events_csv))
display(nb.preview_csv(paths.removed_midnight_csv))
